<a href="https://colab.research.google.com/github/potat0w/Aaagh-more-math/blob/main/deepfake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, BatchNormalization, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.metrics import AUC
from sklearn.utils import shuffle
from PIL import Image

In [4]:
root = '/kaggle/input/140k-real-and-fake-faces'

train_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train'
val_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid'
test_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test'

print("Train_dir Subfolders: ", os.listdir(train_dir))
print("Valid_dir Subfolders: ", os.listdir(val_dir))
print("Test_dir Subfolders: ", os.listdir(test_dir))

train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)
target_size = (256,256)
batch_size = 32

# Load data from directories
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary'
)
val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary'
)


test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=target_size,
    batch_size=batch_size,
    class_mode='binary',
    shuffle=False  # Ensure test data is not shuffled
)


Train_dir Subfolders:  ['fake', 'real']
Valid_dir Subfolders:  ['fake', 'real']
Test_dir Subfolders:  ['fake', 'real']
Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.


In [3]:
class DeepfakeDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images = []

        for label, folder in enumerate(["real", "fake"]):
            folder_path = os.path.join(root_dir, folder)

            for file in os.listdir(folder_path):
                self.images.append(
                    (os.path.join(folder_path, file), label)
                )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        return img, label

NameError: name 'Dataset' is not defined

In [ ]:
import tensorflow as tf
import os
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import (Input, Dense, GlobalAveragePooling2D,
                                     BatchNormalization, Dropout, Multiply, Reshape, Lambda)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC

# Input shape
image_input = Input(shape=(256, 256, 3))

# Load DenseNet121 as the base model
base_model = DenseNet121(weights="imagenet", include_top=False, input_tensor=image_input)

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Squeeze-and-Excitation Block
se = GlobalAveragePooling2D()(base_model.output)
se = Reshape((1, 1, se.shape[1]))(se)
se = Dense(se.shape[-1] // 16, activation='relu')(se)
se = Dense(base_model.output.shape[-1], activation='sigmoid')(se)

# Ensure Multiply layer compatibility
attention_output = Lambda(lambda x: x[0] * x[1])([base_model.output, se])

# Classification head
x = GlobalAveragePooling2D()(attention_output)
x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

# Create model
model = Model(inputs=image_input, outputs=output)

# Compile model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='binary_crossentropy',
              metrics=['accuracy', AUC(name='auc')])

# Print model summary
model.summary()

In [ ]:
# Define dataset directories
train_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train'
val_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/valid'
test_dir = '/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/test'

# Image data generators
train_datagen = ImageDataGenerator(rescale=1./255)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load data
target_size = (256, 256)
batch_size = 32

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=target_size, batch_size=batch_size, class_mode='binary')

val_generator = val_test_datagen.flow_from_directory(
    val_dir, target_size=target_size, batch_size=batch_size, class_mode='binary')

test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=target_size, batch_size=batch_size, class_mode='binary', shuffle=False)

In [ ]:
# Train the model
epochs = 10  # Adjust as needed
history = model.fit(train_generator,
                    validation_data=val_generator,
                    epochs=epochs,
                    verbose=1)

# Save the model
model.save("deepfake_detector.h5")